In [10]:
# ✅ ENV SETUP
import os
from dotenv import load_dotenv

load_dotenv(".env")

FIGMA_TOKEN = os.getenv("FIGMA_TOKEN")
FILE_ID_1 = os.getenv("FIGMA_DOCUMENT_ID1")
FILE_ID_2 = os.getenv("FIGMA_DOCUMENT_ID2")
SAS_URL = os.getenv("SAS_URL")

structure_path = "../03_Outputs/SEA_Modules/en/module_structure.json"
figma_path_1 = "../02_Inputs/figma_jsons/figma_document1.json"
figma_path_2 = "../02_Inputs/figma_jsons/figma_document2.json"
cache_path = "../03_Outputs/image_refs_cache.json"

pages_1 = ["Module 1", "Module 2", "Module 3", "Module 4","Module 9"]
pages_2 = ["Module 5", "Module 6", "Module 7", "Module 8"]

required = ["FIGMA_TOKEN", "FILE_ID_1", "FILE_ID_2", "SAS_URL"]
missing = [v for v in required if not globals()[v]]
if missing:
    raise EnvironmentError(f"❌ Missing env vars: {', '.join(missing)}")

print("✅ Environment loaded.")

# ✅ UTILS
import io
import time
from PIL import Image
from contextlib import contextmanager

✅ Environment loaded.


In [11]:


def format_blob_name(folder: str, name: str) -> str:
    return f"{folder}/{name.replace(':', '_').replace(' ', '_')}.webp"

def format_path(name: str) -> str:
    return name.replace(" ", "_").strip()

@contextmanager
def timed(label):
    t0 = time.time()
    yield
    print(f"⏱️ {label} took {time.time() - t0:.2f}s")

def convert_image_bytes_to_webp(image_bytes: bytes, max_height: int = 1080, quality: int = 95) -> bytes:
    with Image.open(io.BytesIO(image_bytes)) as img:
        img = img.convert("RGB")

        # Calculate scale factor if height is above max_height
        if img.height > max_height:
            scale = max_height / img.height
            new_width = int(img.width * scale)
            new_height = max_height
            img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

        buffer = io.BytesIO()
        img.save(buffer, format="WEBP", quality=quality)
        return buffer.getvalue()


# ✅ AZURE STORAGE
import ssl
from azure.storage.blob.aio import ContainerClient
from azure.storage.blob import ContentSettings
from azure.core.pipeline.transport import AioHttpTransport

USE_INSECURE_SSL = False

def get_container_client():
    if USE_INSECURE_SSL:
        ssl_context = ssl.create_default_context()
        ssl_context.check_hostname = False
        ssl_context.verify_mode = ssl.CERT_NONE
        transport = AioHttpTransport(ssl_context=ssl_context)
        return ContainerClient.from_container_url(SAS_URL, transport=transport)
    return ContainerClient.from_container_url(SAS_URL)

async def get_existing_blob_names(prefix: str = "") -> set[str]:
    blob_names = set()
    async with get_container_client() as client:
        async for blob in client.list_blobs(name_starts_with=prefix):
            blob_names.add(blob.name)
    print(f"📦 {len(blob_names)} blobs found with prefix '{prefix}'")
    return blob_names

async def upload_image_blob(blob_name: str, data: bytes) -> str:
    async with get_container_client() as client:
        blob = client.get_blob_client(blob_name)
        await blob.upload_blob(
            data=data,
            overwrite=True,
            content_settings=ContentSettings(content_type="image/webp"),
        )
        return blob.url.split("?")[0]

def assign_parents(node: dict, parent: dict | None = None):
    node["_parent"] = parent
    for child in node.get("children", []) or []:
        assign_parents(child, node)


# ✅ FETCH IMAGE URLS
import requests
import json

REFS_PATH = "figma_image_refs.json"

def calculate_optimal_scale(canvas_height: float, target_height: int = 1080, max_scale: float = 4.0) -> float:
    """
    Calculate the optimal scale factor for Figma image export.

    Parameters:
    - canvas_height (float): The height of the image on the canvas.
    - target_height (int): The desired minimum height in pixels after scaling.
    - max_scale (float): The maximum scale Figma API allows.

    Returns:
    - float: A scale factor (rounded to 2 decimals) to use in the image export URL.
    """
    if not canvas_height or canvas_height <= 0:
        return 2.0  # sensible default if height is missing

    scale = target_height / canvas_height
    adjusted_scale=round(min(max_scale, max(1.0, scale)), 2)
    return adjusted_scale


def fetch_image_url_single(file_id: str, node_id: str, canvas_height: float, format="png") -> str | None:
    try:
        scale = calculate_optimal_scale(canvas_height)
        res = requests.get(
            f"https://api.figma.com/v1/images/{file_id}",
            headers={"X-Figma-Token": FIGMA_TOKEN},
            params={"ids": node_id, "format": format, "scale": scale},
            timeout=30,
        )
        res.raise_for_status()
        return res.json().get("images", {}).get(node_id)
    except Exception as e:
        print(f"❌ Error fetching URL for {node_id}: {e}")
        return None


def load_image_refs() -> dict[str, str]:
    if os.path.exists(REFS_PATH):
        with open(REFS_PATH, "r") as f:
            return json.load(f)
    return {}

def save_image_refs(refs: dict[str, str]):
    with open(REFS_PATH, "w") as f:
        json.dump(refs, f, indent=2)

# ✅ FETCH IMAGE CONTENT
import aiohttp

async def fetch_image_content(url: str) -> bytes | None:
    try:
        async with aiohttp.ClientSession() as session:
            async with session.get(url, timeout=30) as resp:
                if resp.status == 200:
                    return await resp.read()
                print(f"⚠️ Failed to fetch {url} (status {resp.status})")
    except Exception as e:
        print(f"❌ Error fetching image: {e}")
    return None

# ✅ SCAN FIGMA PAGE

from math import ceil

def extract_generic_image(node: dict, page_name: str) -> dict | None:
    if node.get("type") != "RECTANGLE":
        return None
    for fill in node.get("fills", []):
        if fill.get("type") == "IMAGE" and "imageRef" in fill:
            canvas_height = node.get("absoluteBoundingBox", {}).get("height", 0)
            return {
                "node_id": node["id"],
                "blob_name": format_blob_name(f"Modules/{format_path(page_name)}", node["id"]),
                "image_ref": fill["imageRef"],
                "canvas_height": node.get("absoluteBoundingBox", {}).get("height", 0),
                "type": "generic"
            }
    return None


def scan_figma_page(page):
    scanned = {}
    assign_parents(page)
    type_counts = {"generic": 0}

    def walk(node):
        if node.get("type") == "RECTANGLE":
            result = extract_generic_image(node, page["name"])
            if result:
                scanned[result["node_id"]] = result
                type_counts["generic"] += 1
        for child in node.get("children", []) or []:
            walk(child)

    walk(page)
    total = sum(type_counts.values())
    print(f"🔍 {page['name']}: {total} image rectangles found")
    return scanned


# ✅ UPLOAD
async def upload_figma_images(file_id, id_to_blobname, image_refs, existing_blobs, cache_path, overwrite=False):
    previous_refs = {}
    if os.path.exists(cache_path):
        try:
            with open(cache_path, "r") as f:
                previous_refs = json.load(f)
        except Exception as e:
            print(f"❌ Failed to load cache: {e}")

    to_upload = {
        node_id: blob_name
        for node_id, blob_name in id_to_blobname.items()
        if overwrite or blob_name not in existing_blobs or image_refs.get(node_id) != previous_refs.get(node_id)
    }

    print(f"🧮 {len(id_to_blobname)} total | {len(to_upload)} to upload")

    uploaded = 0
    for i, (node_id, blob_name) in enumerate(to_upload.items(), 1):
        canvas_height = scanned[node_id].get("canvas_height", 0)
        url = fetch_image_url_single(file_id, node_id, canvas_height)
        if not url:
            continue
        image_bytes = await fetch_image_content(url)
        if not image_bytes:
            continue
        webp = convert_image_bytes_to_webp(image_bytes)
        await upload_image_blob(blob_name, webp)
        print(f"✅ [{i}] Uploaded {blob_name}")
        previous_refs[node_id] = image_refs[node_id]
        try:
            with open(cache_path, "w") as f:
                json.dump(previous_refs, f, indent=2)
        except Exception as e:
            print(f"❌ Failed to update cache after {blob_name}: {e}")
        uploaded += 1

    print(f"📊 Done | Uploaded: {uploaded} | Skipped: {len(id_to_blobname) - uploaded}")


with open(structure_path) as f:
    structure_json = json.load(f)
with open(figma_path_1) as f:
    figma_data_1 = json.load(f)
with open(figma_path_2) as f:
    figma_data_2 = json.load(f)

file_page_sets = [
    (FILE_ID_1, figma_data_1["children"], pages_1),
    (FILE_ID_2, figma_data_2["children"], pages_2)
]


print("📦 Fetching existing Azure blobs...")
existing_blobs = await get_existing_blob_names()

print("🔄 Loading cached image references...")
updated_refs = load_image_refs()

📦 Fetching existing Azure blobs...
📦 6330 blobs found with prefix ''
🔄 Loading cached image references...


In [3]:
# ✅ 8. MAIN PIPELINE RUNNER
import json




for file_id, figma_pages, page_names in file_page_sets:
    for page_name in page_names:
        page = next((p for p in figma_pages if p["name"] == page_name), None)
        if not page:
            print(f"⚠️ Page not found: {page_name}")
            continue

        print(f"\n📄 Processing page: {page_name}")
        scanned = scan_figma_page(page)
        if not scanned:
            continue

        id_to_blobname = {
            node_id: info["blob_name"].replace(" ", "_")
            for node_id, info in scanned.items()
        }

        fetched_refs = {
            node_id: info["image_ref"]
            for node_id, info in scanned.items()
            if "image_ref" in info
        }

        await upload_figma_images(
            file_id=file_id,
            id_to_blobname=id_to_blobname,
            image_refs=fetched_refs,
            existing_blobs=existing_blobs,
            cache_path=cache_path,
            overwrite=True,
        )

print("✅ Upload pipeline complete.")


📄 Processing page: Module 1
🔍 Module 1: 189 image rectangles found
🧮 189 total | 189 to upload
✅ [1] Uploaded Modules/Module_1/609_345.webp
✅ [2] Uploaded Modules/Module_1/488_1680.webp
✅ [3] Uploaded Modules/Module_1/488_1685.webp
✅ [4] Uploaded Modules/Module_1/488_1690.webp
✅ [5] Uploaded Modules/Module_1/3661_206.webp
✅ [6] Uploaded Modules/Module_1/488_2078.webp
✅ [7] Uploaded Modules/Module_1/488_2070.webp
✅ [8] Uploaded Modules/Module_1/3623_76.webp
✅ [9] Uploaded Modules/Module_1/3623_67.webp
✅ [10] Uploaded Modules/Module_1/522_537.webp
✅ [11] Uploaded Modules/Module_1/3601_191.webp
✅ [12] Uploaded Modules/Module_1/3601_196.webp
✅ [13] Uploaded Modules/Module_1/3601_202.webp
✅ [14] Uploaded Modules/Module_1/488_1667.webp
✅ [15] Uploaded Modules/Module_1/613_471.webp
✅ [16] Uploaded Modules/Module_1/488_1646.webp
✅ [17] Uploaded Modules/Module_1/488_1622.webp
✅ [18] Uploaded Modules/Module_1/488_1601.webp
✅ [19] Uploaded Modules/Module_1/488_993.webp
✅ [20] Uploaded Modules/Mo

In [21]:
# ✅ INFOGRAPHIC EXPORT CELL (clean logging, overwrite=True, no intermediate prints)

import re
import io
from contextlib import redirect_stdout

def find_figma_page(figma_pages: list[dict], page_name: str) -> dict | None:
    for p in figma_pages:
        if p.get("name") == page_name and p.get("type") in ("CANVAS", "PAGE", "FRAME"):
            return p
    return None

def find_first_node_by_name(node: dict, target_name: str) -> dict | None:
    if node.get("name") == target_name:
        return node
    for ch in node.get("children", []) or []:
        hit = find_first_node_by_name(ch, target_name)
        if hit:
            return hit
    return None

def walk_collect(node: dict, out: list):
    out.append(node)
    for ch in node.get("children", []) or []:
        walk_collect(ch, out)

total_uploaded = 0
total_skipped  = 0
per_item_logs  = []

for file_id, figma_pages, page_names in file_page_sets:
    for page_name in page_names:
        page = find_figma_page(figma_pages, page_name)
        if not page:
            continue

        nodes = []
        walk_collect(page, nodes)

        infographic_frames = [n for n in nodes if n.get("type") == "FRAME" and n.get("name") == "infographic"]
        if not infographic_frames:
            continue

        for inf in infographic_frames:
            design = find_first_node_by_name(inf, "infographic-design")
            if not design:
                total_skipped += 1
                per_item_logs.append(f"⏭️ Skipped Modules/{format_path(page_name)}/(missing infographic-design)")
                continue

            node_id = design["id"]
            blob_name = f"Modules/{format_path(page_name)}/{node_id.replace(':','_')}.webp"

            id_to_blob = {node_id: blob_name}
            image_refs = {node_id: None}  # infographic-design is a FRAME; caching not applicable here

            buf = io.StringIO()
            with redirect_stdout(buf):
                # ✅ Call positionally to avoid keyword-name mismatch
                await upload_figma_images(
            file_id=file_id,
            id_to_blobname=id_to_blob,
            image_refs=image_refs,
            existing_blobs=existing_blobs,
            cache_path=cache_path,
            overwrite=True,
                )

            raw = buf.getvalue()

            # Parse the last "Done | Uploaded: X | Skipped: Y"
            m = re.findall(r"Done\s*\|\s*Uploaded:\s*(\d+)\s*\|\s*Skipped:\s*(\d+)", raw)
            if m:
                up, sk = map(int, m[-1])
            else:
                # Fallback heuristics
                up = 1 if ("Uploaded" in raw and blob_name in raw) else 0
                sk = 1 - up

            total_uploaded += up
            total_skipped  += sk

            if up > 0:
                per_item_logs.append(f"✅ Uploaded {blob_name}")
            else:
                per_item_logs.append(f"⏭️ Skipped {blob_name}")

# Output: only per-item + final summary (no intermediate logs)
for line in per_item_logs:
    print(line)

print(f"🧾 Infographics: uploaded {total_uploaded}, skipped {total_skipped}")
print("✅ Infographic export complete.")


⏭️ Skipped Modules/Module_1/(missing infographic-design)
⏭️ Skipped Modules/Module_1/(missing infographic-design)
⏭️ Skipped Modules/Module_1/(missing infographic-design)
⏭️ Skipped Modules/Module_1/(missing infographic-design)
⏭️ Skipped Modules/Module_1/(missing infographic-design)
⏭️ Skipped Modules/Module_1/(missing infographic-design)
⏭️ Skipped Modules/Module_1/(missing infographic-design)
⏭️ Skipped Modules/Module_1/(missing infographic-design)
⏭️ Skipped Modules/Module_1/(missing infographic-design)
⏭️ Skipped Modules/Module_1/(missing infographic-design)
⏭️ Skipped Modules/Module_1/(missing infographic-design)
⏭️ Skipped Modules/Module_2/(missing infographic-design)
⏭️ Skipped Modules/Module_2/(missing infographic-design)
⏭️ Skipped Modules/Module_2/(missing infographic-design)
⏭️ Skipped Modules/Module_2/(missing infographic-design)
⏭️ Skipped Modules/Module_2/(missing infographic-design)
⏭️ Skipped Modules/Module_2/(missing infographic-design)
⏭️ Skipped Modules/Module_2/(mi

In [13]:
# ✅ STANDALONE: EXCEL PHOTO INDEX (2 FIGMA DOCS) — TEST MODE (default 5 images)
# Run AFTER your ENV SETUP cell (the one that defines FILE_ID_1/2, figma_path_1/2, pages_1/2, and loads FIGMA_TOKEN).
# This cell:
#  - Loads cached Figma JSONs (figma_path_1, figma_path_2)
#  - Walks selected pages across both docs
#  - Finds RECTANGLE layers with IMAGE fills
#  - Derives module/chapter/lesson from PAGE + SECTION ancestry
#  - Fetches image bytes via your existing async fetchers (fetch_image_url_single + fetch_image_content)
#  - Writes an Excel with embedded thumbnails in Column A + metadata (including photo layer name)

import os, re, json
from io import BytesIO
from PIL import Image
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage

# =========================
# 🔧 SWITCHES YOU CHANGE
# =========================
TEST_MODE = True                 # True = quick test, False = full run
TEST_LIMIT = None                   # how many images to process in test mode
EXCEL_OUT_PATH = "../03_Outputs/photo_index_thumbs.xlsx"

THUMB_MAX_PX   = (220, 220)      # thumbnails are column A; keep them modest
THUMB_QUALITY  = 70
FIGMA_SCALE    = 2               # note: in your notebook, fetch_image_url_single uses (file_id, node_id, canvas_height)
# =========================


# ---------- Load cached Figma JSONs ----------
with open(figma_path_1, "r") as f:
    figma_data_1 = json.load(f)
with open(figma_path_2, "r") as f:
    figma_data_2 = json.load(f)

file_page_sets = [
    (FILE_ID_1, figma_data_1["children"], pages_1),
    (FILE_ID_2, figma_data_2["children"], pages_2),
]


# ---------- Utilities ----------
def px_to_excel_row_height(px: int) -> float:
    return px * 0.75  # px -> points (approx)

def px_to_excel_col_width(px: int) -> float:
    return px / 7.0   # rough but fine

def make_jpeg_thumbnail_bytes(image_bytes: bytes, max_px=(220,220), quality=70):
    with Image.open(BytesIO(image_bytes)) as im:
        im = im.convert("RGB")
        im.thumbnail(max_px)
        bio = BytesIO()
        im.save(bio, format="JPEG", quality=quality, optimize=True, progressive=True)
        bio.seek(0)
        return bio, im.size


# ---------- Hierarchy helpers ----------
def assign_parents(node: dict, parent=None):
    node["_parent"] = parent
    for c in node.get("children", []) or []:
        assign_parents(c, node)

def walk(node):
    yield node
    for ch in node.get("children", []) or []:
        yield from walk(ch)

def find_nearest_ancestor(node, types):
    cur = node.get("_parent")
    while cur:
        if cur.get("type") in types:
            return cur
        cur = cur.get("_parent")
    return None

def find_next_ancestor_section(start_node):
    cur = start_node.get("_parent") if start_node else None
    while cur:
        if cur.get("type") == "SECTION":
            return cur
        cur = cur.get("_parent")
    return None

def parse_module_number(name):
    if not name: return None
    m = re.search(r"\bModule\s*0*(\d+)\b", name, re.IGNORECASE)
    if m: return m.group(1)
    m = re.search(r"\bM\s*0*(\d+)\b", name, re.IGNORECASE)
    if m: return m.group(1)
    return None

def parse_chapter_number(name):
    if not name: return None
    m = re.search(r"\bChapter\s*0*(\d+)\b", name, re.IGNORECASE)
    if m: return m.group(1)
    m = re.search(r"\bC\s*0*(\d+)\b", name, re.IGNORECASE)
    if m: return m.group(1)
    return None

def parse_lesson_number(name):
    if not name: return None
    m = re.search(r"\bLesson\s*0*(\d+)\b", name, re.IGNORECASE)
    if m: return m.group(1)
    m = re.search(r"\bL\s*0*(\d+)\b", name, re.IGNORECASE)
    if m: return m.group(1)
    return None

def find_lesson_section(node):
    # nearest section might be a subsection; walk up until it looks like a lesson section
    sec = find_nearest_ancestor(node, ("SECTION",))
    while sec and not parse_lesson_number(sec.get("name","")):
        sec = find_next_ancestor_section(sec)
    return sec

def find_chapter_section_from_lesson(lesson_section):
    sec = find_next_ancestor_section(lesson_section) if lesson_section else None
    while sec and not parse_chapter_number(sec.get("name","")):
        sec = find_next_ancestor_section(sec)
    return sec

def is_image_rectangle(node):
    if node.get("type") != "RECTANGLE":
        return None
    for fill in node.get("fills") or []:
        if fill.get("type") == "IMAGE" and fill.get("imageRef"):
            return fill["imageRef"]
    return None

def find_page(figma_pages, name):
    for p in figma_pages:
        if p.get("name") == name:
            return p
    return None


# ---------- Workbook ----------
wb = Workbook()
ws = wb.active
ws.title = "PhotoIndex"

# Header row: THUMBNAIL IS COLUMN 1 (A)
ws.append([
    "thumbnail",            # A (embedded image)
    "file_id",              # B
    "module_number",        # C
    "chapter_number",       # D
    "lesson_number",        # E
    "page_name",            # F
    "chapter_section_name", # G
    "lesson_section_name",  # H
    "frame_name",           # I
    "photo_layer_name",     # J  ✅ NEW
    "node_id",              # K
    "image_ref",            # L
    "figma_image_url",      # M
])

# widths
ws.column_dimensions["A"].width = px_to_excel_col_width(THUMB_MAX_PX[0] + 14)  # thumbnail col
ws.column_dimensions["B"].width = 18
ws.column_dimensions["C"].width = 14
ws.column_dimensions["D"].width = 14
ws.column_dimensions["E"].width = 14
ws.column_dimensions["F"].width = 22
ws.column_dimensions["G"].width = 48
ws.column_dimensions["H"].width = 48
ws.column_dimensions["I"].width = 30
ws.column_dimensions["J"].width = 36
ws.column_dimensions["K"].width = 22
ws.column_dimensions["L"].width = 18
ws.column_dimensions["M"].width = 60

row_ptr = 2


# ---------- Runner (async) ----------
import asyncio

async def export_photo_index_excel():
    global row_ptr

    limit = TEST_LIMIT if TEST_MODE else None
    processed = 0
    skipped = 0

    print(f"🧪 TEST_MODE={TEST_MODE} | limit={limit if limit is not None else 'FULL'}")

    for file_id, figma_pages, page_names in file_page_sets:
        for page_name in page_names:
            page = find_page(figma_pages, page_name)
            if not page:
                print(f"⚠️ Page not found: {page_name}")
                continue

            assign_parents(page)

            module_number = parse_module_number(page_name)
            print(f"\n📄 {page_name} | module={module_number} | file={file_id}")

            for node in walk(page):
                image_ref = is_image_rectangle(node)
                if not image_ref:
                    continue

                node_id = node["id"]
                photo_layer_name = node.get("name")  # ✅ photo layer name

                # Your notebook’s fetch_image_url_single signature is: (file_id, node_id, canvas_height)
                canvas_height = int((node.get("absoluteBoundingBox") or {}).get("height") or 0) or 600
                url = fetch_image_url_single(file_id, node_id, canvas_height)
                if not url:
                    skipped += 1
                    continue

                img_bytes = await fetch_image_content(url)
                if not img_bytes:
                    skipped += 1
                    continue

                # hierarchy info
                lesson_section = find_lesson_section(node)
                chapter_section = find_chapter_section_from_lesson(lesson_section)

                lesson_section_name = lesson_section.get("name") if lesson_section else None
                chapter_section_name = chapter_section.get("name") if chapter_section else None

                chapter_number = parse_chapter_number(chapter_section_name or "")
                lesson_number  = parse_lesson_number(lesson_section_name or "")

                frame = find_nearest_ancestor(node, ("FRAME",))
                frame_name = frame.get("name") if frame else None

                # thumbnail
                thumb_bio, (tw, th) = make_jpeg_thumbnail_bytes(
                    img_bytes, max_px=THUMB_MAX_PX, quality=THUMB_QUALITY
                )

                # Write metadata (thumbnail is column A but embedded, so we write other fields starting col B)
                ws.cell(row_ptr, 2, file_id)
                ws.cell(row_ptr, 3, module_number)
                ws.cell(row_ptr, 4, chapter_number)
                ws.cell(row_ptr, 5, lesson_number)
                ws.cell(row_ptr, 6, page_name)
                ws.cell(row_ptr, 7, chapter_section_name)
                ws.cell(row_ptr, 8, lesson_section_name)
                ws.cell(row_ptr, 9, frame_name)
                ws.cell(row_ptr, 10, photo_layer_name)
                ws.cell(row_ptr, 11, node_id)
                ws.cell(row_ptr, 12, image_ref)
                ws.cell(row_ptr, 13, url)

                # Embed thumbnail in column A
                xl_img = XLImage(thumb_bio)
                xl_img.width, xl_img.height = tw, th
                xl_img.anchor = f"A{row_ptr}"
                ws.add_image(xl_img)
                ws.row_dimensions[row_ptr].height = px_to_excel_row_height(th + 6)

                print(f"  ✅ [{processed+1}{f'/{limit}' if limit else ''}] "
                      f"M{module_number} C{chapter_number} L{lesson_number} | "
                      f"layer='{photo_layer_name or ''}' | frame='{frame_name or ''}'")

                row_ptr += 1
                processed += 1

                if limit is not None and processed >= limit:
                    os.makedirs(os.path.dirname(EXCEL_OUT_PATH), exist_ok=True)
                    wb.save(EXCEL_OUT_PATH)
                    print(f"\n✅ Stopped after {processed} images (test). Saved: {EXCEL_OUT_PATH}")
                    print(f"   Skipped: {skipped}")
                    return

    os.makedirs(os.path.dirname(EXCEL_OUT_PATH), exist_ok=True)
    wb.save(EXCEL_OUT_PATH)
    print(f"\n✅ Finished FULL run. Processed: {processed} | Skipped: {skipped}")
    print(f"📎 Saved: {EXCEL_OUT_PATH}")

await export_photo_index_excel()


🧪 TEST_MODE=True | limit=FULL

📄 Module 1 | module=1 | file=qyIFc7kcwzjfrlmWs0BBdG
  ✅ [1] M1 CNone LNone | layer='image_34' | frame='chapter_outro'
  ✅ [2] M1 C4 L3 | layer='image_15' | frame='key_takeaways'
  ✅ [3] M1 C4 L3 | layer='image_8' | frame='key_takeaways'
  ✅ [4] M1 C4 L3 | layer='image_23' | frame='key_takeaways'
  ✅ [5] M1 C4 L3 | layer='src' | frame='photo-vertical'
  ✅ [6] M1 C4 L3 | layer='image_70' | frame='lesson_subpart_cover'
  ✅ [7] M1 C4 L3 | layer='image_71' | frame='lesson_subpart_cover'
  ✅ [8] M1 C4 L3 | layer='src' | frame='photo-vertical'
  ✅ [9] M1 C4 L3 | layer='src' | frame='lesson_part_cover'
  ✅ [10] M1 C4 L3 | layer='image_46' | frame='lesson_cover'
  ✅ [11] M1 C4 L2 | layer='image_22' | frame='key_takeaways'
  ✅ [12] M1 C4 L2 | layer='image_40' | frame='key_takeaways'
  ✅ [13] M1 C4 L2 | layer='image_3' | frame='key_takeaways'
  ✅ [14] M1 C4 L2 | layer='image_62' | frame='photo-vertical'
  ✅ [15] M1 C4 L2 | layer='image_61' | frame='lesson_subpart_co